# Notebook 03: Model Training — Statistical & ML Forecasting

## CRISP-DM Phase: Modeling

**Project**: OptiWMS — AI-Driven Warehouse Management System  
**Reference**: Petropoulos et al. (2022), Sections 2.3 (Exponential Smoothing), 2.5 (ARIMA), 2.7 (ML), 2.8 (Intermittent)

---

### Model Strategy (Enterprise-Grade)

| Model Type | Method | Use Case | Quantile Output |
|------------|--------|----------|----------------|
| Statistical | ETS (Holt-Winters) | Smooth seasonal demand | p10, p50, p90 via residual std |
| Statistical | ARIMA/SARIMAX | Trending + seasonal | p10, p50, p90 from confidence intervals |
| Statistical | Croston (SBA) | Intermittent demand (RM) | p50 + bootstrap intervals |
| ML | LightGBM | Large-scale, fast, handles categoricals | Quantile regression |
| ML | CatBoost | Excellent with categoricals | Quantile regression |
| ML | XGBoost | Robust baseline | Quantile regression |
| Baseline | Seasonal Naive | Repeat same month last year | - |

### Why ML for Warehousing?

> *"ML methods are particularly suited when cross-learning from a large number of related time series is beneficial."*  
> — Petropoulos et al. (2022), Section 2.7.4  
> *"The M5 competition confirmed that gradient-boosted trees outperform traditional methods on complex, hierarchical data."*

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import time

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_SM = True
except ImportError:
    HAS_SM = False

try:
    import mlflow
    import mlflow.sklearn
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False
    print('MLflow not installed — results will be logged locally only')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
SEED = 42
np.random.seed(SEED)

ROOT = Path('..').resolve()
DATA_DIR = ROOT.parent / 'Forecast model train data optiwms'
GEN_DIR  = ROOT / 'outputs' / 'generated'
ENG_DIR  = ROOT / 'outputs' / 'engineered'

print('Libraries loaded successfully')

In [ ]:
# Load engineered features from NB02
if (ENG_DIR / 'fg_features_engineered.csv').exists():
    fg = pd.read_csv(ENG_DIR / 'fg_features_engineered.csv')
    fg['month'] = pd.to_datetime(fg['month'])
    print(f'Loaded engineered features: {fg.shape}')
else:
    print('Run NB02 first to create engineered features. Loading raw data instead...')
    fg = pd.read_csv(DATA_DIR / 'hemas_scenario_c_dataset_cleaned.csv')
    fg['month'] = pd.to_datetime(fg['month'])
    if 'demand_units_clean' in fg.columns:
        fg['demand_units'] = fg['demand_units_clean']
    fg = fg.sort_values(['fg_code', 'month']).reset_index(drop=True)

print(f'SKUs: {fg["fg_code"].nunique()}, Rows: {len(fg):,}')

## 1. MLflow Experiment Setup

> *"Experiment tracking is essential for reproducibility."* — ML Pipelines Module, Week 3  
> *"MLflow provides model versioning, comparison, and deployment."* — MLflow Module, Week 6

In [ ]:
# MLflow setup
MLFLOW_TRACKING_URI = 'http://localhost:5001'
EXPERIMENT_NAME = 'optiwms-demand-forecast-v6'

if HAS_MLFLOW:
    try:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.set_experiment(EXPERIMENT_NAME)
        print(f'MLflow tracking at: {MLFLOW_TRACKING_URI}')
        print(f'Experiment: {EXPERIMENT_NAME}')
    except Exception as e:
        print(f'MLflow server not reachable ({e}). Logging locally.')
        HAS_MLFLOW = False
else:
    print('MLflow not available — running without tracking')

In [ ]:
# Metric helper functions
def wape(y_true, y_pred):
    """Weighted Absolute Percentage Error — standard for demand forecasting."""
    return np.sum(np.abs(y_true - y_pred)) / max(np.sum(np.abs(y_true)), 1)

def mase(y_true, y_pred, y_train, seasonality=12):
    """Mean Absolute Scaled Error (Hyndman & Koehler, 2006).
    Referenced in Petropoulos Section 2.12.2 as the recommended scale-free metric."""
    n = len(y_train)
    naive_errors = np.abs(y_train[seasonality:] - y_train[:-seasonality])
    scale = np.mean(naive_errors) if len(naive_errors) > 0 else 1.0
    if scale == 0:
        scale = 1.0
    return np.mean(np.abs(y_true - y_pred)) / scale

def bias_metric(y_true, y_pred):
    """Signed bias — positive means over-forecasting."""
    return np.mean(y_pred - y_true)

def compute_all_metrics(y_true, y_pred, y_train=None):
    metrics = {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'WAPE': wape(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'Bias': bias_metric(y_true, y_pred),
    }
    if y_train is not None and len(y_train) > 12:
        metrics['MASE'] = mase(y_true, y_pred, y_train)
    return metrics

print('Metric functions defined: RMSE, MAE, WAPE, R2, Bias, MASE')

## 2. Time-Series Split for Training

In [ ]:
# Temporal split
sorted_months = sorted(fg['month'].unique())
n_months = len(sorted_months)
train_end = sorted_months[min(23, n_months-7)]
val_end = sorted_months[min(29, n_months-1)]

train_df = fg[fg['month'] <= train_end].copy()
val_df   = fg[(fg['month'] > train_end) & (fg['month'] <= val_end)].copy()
test_df  = fg[fg['month'] > val_end].copy()

# Features for ML models
feature_cols = [
    'month_num', 'quarter', 'year', 'is_year_end', 'is_sl_peak',
    'month_sin', 'month_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'demand_lag_6', 'demand_lag_12',
    'demand_rmean_3', 'demand_rmean_6', 'demand_rstd_3', 'demand_rstd_6',
    'demand_rmin_3', 'demand_rmax_3', 'demand_rmin_6', 'demand_rmax_6',
    'demand_cv_6', 'demand_momentum',
    'fg_category_enc', 'fg_code_enc',
]
for col in ['on_hand_inventory', 'lead_time_days', 'supplier_otif',
            'promotion_flag', 'holiday_flag', 'price_per_unit']:
    if col in fg.columns:
        feature_cols.append(col)

available_feats = [c for c in feature_cols if c in fg.columns]
TARGET = 'demand_units'

X_train = train_df[available_feats].values
y_train = train_df[TARGET].values
X_val = val_df[available_feats].values
y_val = val_df[TARGET].values

if len(test_df) > 0:
    X_test = test_df[available_feats].values
    y_test = test_df[TARGET].values
else:
    X_test, y_test = X_val, y_val  # fallback

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Features: {len(available_feats)}')

## 3. Statistical Models

### 3.1 Seasonal Naive Baseline

> *"Always start with a naive or seasonal naive benchmark. Any model that cannot outperform it is not worth deploying."*  
> — Petropoulos et al. (2022), Section 2.3.1

In [ ]:
# 3.1 Seasonal Naive (same month last year)
snaive_preds = []
snaive_actuals = []

for sku in val_df['fg_code'].unique():
    sku_train = train_df[train_df['fg_code'] == sku].sort_values('month')
    sku_val = val_df[val_df['fg_code'] == sku].sort_values('month')
    
    for _, row in sku_val.iterrows():
        same_month_ly = sku_train[sku_train['month'].dt.month == row['month'].month]
        if len(same_month_ly) > 0:
            pred = same_month_ly['demand_units'].iloc[-1]
        else:
            pred = sku_train['demand_units'].mean()
        snaive_preds.append(pred)
        snaive_actuals.append(row['demand_units'])

snaive_metrics = compute_all_metrics(np.array(snaive_actuals), np.array(snaive_preds), y_train)

print('=== Seasonal Naive Baseline ===')
for k, v in snaive_metrics.items():
    print(f'  {k}: {v:.4f}')

results_log = [{'model': 'Seasonal Naive', **snaive_metrics, 'type': 'Baseline'}]

In [ ]:
# 3.2 ETS (Holt-Winters) — Petropoulos Section 2.3
if HAS_SM:
    ets_preds = []
    ets_actuals = []
    n_sku_ets = 0
    
    # Fit per-SKU ETS (sample 20 SKUs for speed)
    sample_skus = train_df.groupby('fg_code')['demand_units'].mean().nlargest(20).index
    
    for sku in sample_skus:
        try:
            sku_df = train_df.loc[train_df['fg_code'] == sku, ['month', 'demand_units']].copy()
            sku_df['month'] = sku_df['month'].dt.to_period('M').dt.to_timestamp()
            sku_train = sku_df.groupby('month')['demand_units'].sum().sort_index().asfreq('MS')
            sku_val = val_df[val_df['fg_code'] == sku].sort_values('month')
            
            if len(sku_train) < 24 or len(sku_val) == 0:
                continue
            
            model = ExponentialSmoothing(
                sku_train, seasonal_periods=12,
                trend='add', seasonal='add',
                damped_trend=True
            ).fit(optimized=True)
            
            forecast = model.forecast(len(sku_val))
            ets_preds.extend(forecast.values)
            ets_actuals.extend(sku_val['demand_units'].values)
            n_sku_ets += 1
        except Exception:
            continue
    
    if len(ets_preds) > 0:
        ets_metrics = compute_all_metrics(np.array(ets_actuals), np.array(ets_preds), y_train)
        print(f'=== ETS (Holt-Winters Additive Damped) — {n_sku_ets} SKUs ===')
        for k, v in ets_metrics.items():
            print(f'  {k}: {v:.4f}')
        results_log.append({'model': 'ETS (HW Damped)', **ets_metrics, 'type': 'Statistical'})
else:
    print('statsmodels not available — skipping ETS')

In [ ]:
# 3.3 SARIMAX — Petropoulos Section 2.5
if HAS_SM:
    sarima_preds = []
    sarima_actuals = []
    n_sku_sarima = 0
    
    for sku in sample_skus[:10]:  # 10 SKUs for speed
        try:
            sku_df = train_df.loc[train_df['fg_code'] == sku, ['month', 'demand_units']].copy()
            sku_df['month'] = sku_df['month'].dt.to_period('M').dt.to_timestamp()
            sku_train = sku_df.groupby('month')['demand_units'].sum().sort_index().asfreq('MS')
            sku_val = val_df[val_df['fg_code'] == sku].sort_values('month')
            
            if len(sku_train) < 24 or len(sku_val) == 0:
                continue
            
            model = SARIMAX(sku_train, order=(1,1,1), seasonal_order=(1,1,0,12),
                           enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=200)
            forecast = fit.forecast(len(sku_val))
            
            sarima_preds.extend(forecast.values)
            sarima_actuals.extend(sku_val['demand_units'].values)
            n_sku_sarima += 1
        except Exception:
            continue
    
    if len(sarima_preds) > 0:
        sarima_metrics = compute_all_metrics(np.array(sarima_actuals), np.clip(np.array(sarima_preds), 0, None), y_train)
        print(f'=== SARIMAX(1,1,1)(1,1,0,12) — {n_sku_sarima} SKUs ===')
        for k, v in sarima_metrics.items():
            print(f'  {k}: {v:.4f}')
        results_log.append({'model': 'SARIMAX', **sarima_metrics, 'type': 'Statistical'})
else:
    print('statsmodels not available — skipping SARIMAX')

## 4. ML Models — Gradient Boosted Trees

> *"Gradient boosted trees have become the de-facto standard for tabular forecasting tasks [...] The M5 winners used LightGBM and achieved substantial improvements over statistical benchmarks."*  
> — Petropoulos et al. (2022), Section 2.7.4

### Why Tree-Based Models for Warehousing?
- Handle non-linear relationships (promotions, holidays)
- Native categorical support (CatBoost)
- Cross-learn across hundreds of SKUs simultaneously
- Quantile regression for uncertainty intervals (p10/p50/p90)

In [ ]:
# 4.1 LightGBM (M5 competition winner architecture)
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': 8,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 500,
    'verbose': -1,
    'random_state': SEED
}

t0 = time.time()
lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
lgb_time = time.time() - t0

lgb_pred_val = np.clip(lgb_model.predict(X_val), 0, None)
lgb_metrics = compute_all_metrics(y_val, lgb_pred_val, y_train)

print(f'=== LightGBM (Validation) — trained in {lgb_time:.1f}s ===')
for k, v in lgb_metrics.items():
    print(f'  {k}: {v:.4f}')
print(f'  Best iteration: {lgb_model.best_iteration_}')

results_log.append({'model': 'LightGBM', **lgb_metrics, 'type': 'ML', 'train_time': lgb_time})

# Log to MLflow
if HAS_MLFLOW:
    with mlflow.start_run(run_name='LightGBM-v6'):
        mlflow.log_params(lgb_params)
        mlflow.log_metrics({k: v for k, v in lgb_metrics.items()})
        mlflow.log_metric('train_time_sec', lgb_time)
        mlflow.sklearn.log_model(lgb_model, 'model')
        print('  Logged to MLflow')

In [ ]:
# 4.2 CatBoost
cat_params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3,
    'random_seed': SEED,
    'verbose': 0,
    'early_stopping_rounds': 50
}

t0 = time.time()
cat_model = CatBoostRegressor(**cat_params)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=0)
cat_time = time.time() - t0

cat_pred_val = np.clip(cat_model.predict(X_val), 0, None)
cat_metrics = compute_all_metrics(y_val, cat_pred_val, y_train)

print(f'=== CatBoost (Validation) — trained in {cat_time:.1f}s ===')
for k, v in cat_metrics.items():
    print(f'  {k}: {v:.4f}')

results_log.append({'model': 'CatBoost', **cat_metrics, 'type': 'ML', 'train_time': cat_time})

if HAS_MLFLOW:
    with mlflow.start_run(run_name='CatBoost-v6'):
        mlflow.log_params({k: str(v) for k, v in cat_params.items()})
        mlflow.log_metrics({k: v for k, v in cat_metrics.items()})
        mlflow.log_metric('train_time_sec', cat_time)
        print('  Logged to MLflow')

In [ ]:
# 4.3 XGBoost
xgb_params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.05,
    'max_depth': 8,
    'n_estimators': 500,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': SEED,
    'verbosity': 0,
    'early_stopping_rounds': 50
}

t0 = time.time()
xgb_model = xgb.XGBRegressor(**xgb_params)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_time = time.time() - t0

xgb_pred_val = np.clip(xgb_model.predict(X_val), 0, None)
xgb_metrics = compute_all_metrics(y_val, xgb_pred_val, y_train)

print(f'=== XGBoost (Validation) — trained in {xgb_time:.1f}s ===')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v:.4f}')

results_log.append({'model': 'XGBoost', **xgb_metrics, 'type': 'ML', 'train_time': xgb_time})

if HAS_MLFLOW:
    with mlflow.start_run(run_name='XGBoost-v6'):
        mlflow.log_params({k: str(v) for k, v in xgb_params.items()})
        mlflow.log_metrics({k: v for k, v in xgb_metrics.items()})
        mlflow.log_metric('train_time_sec', xgb_time)
        print('  Logged to MLflow')

## 5. Quantile Regression (Uncertainty Estimation)

Enterprise WMS requires **prediction intervals** (p10, p50, p90) for:
- **p50** — median forecast for slotting velocity
- **p90** — safety stock calculation (serve 90% of demand scenarios)
- **p90 - p10 spread** — demand volatility for GA placement

> *"Probabilistic forecasts are far more useful than point forecasts for inventory management."*  
> — Petropoulos et al. (2022), Section 2.11

In [ ]:
# 5.1 LightGBM Quantile Regression
quantile_models = {}

for q, alpha in [('p10', 0.10), ('p50', 0.50), ('p90', 0.90)]:
    qr_params = {
        'objective': 'quantile',
        'alpha': alpha,
        'metric': 'quantile',
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': 8,
        'n_estimators': 300,
        'verbose': -1,
        'random_state': SEED
    }
    
    model = lgb.LGBMRegressor(**qr_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(30, verbose=False)])
    quantile_models[q] = model

# Generate quantile forecasts
q_preds = {}
for q, model in quantile_models.items():
    q_preds[q] = np.clip(model.predict(X_val), 0, None)

# Visualise quantile forecasts for one SKU
sku_idx = val_df['fg_code'] == val_df['fg_code'].unique()[0]
sku_months = val_df.loc[sku_idx, 'month']
sku_actual = val_df.loc[sku_idx, 'demand_units']

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sku_months, sku_actual, 'ko-', label='Actual', markersize=6)
ax.plot(sku_months, q_preds['p50'][sku_idx], 'b-', label='p50 (median)', linewidth=2)
ax.fill_between(sku_months,
                q_preds['p10'][sku_idx],
                q_preds['p90'][sku_idx],
                alpha=0.3, color='blue', label='p10-p90 interval')
ax.set_title(f'Quantile Forecast — {val_df["fg_code"].unique()[0]}')
ax.set_ylabel('Demand (units)')
ax.legend()
plt.tight_layout()
plt.show()

# Coverage check
in_interval = ((y_val >= q_preds['p10']) & (y_val <= q_preds['p90'])).mean()
print(f'\n80% Prediction Interval Coverage: {in_interval:.1%} (target: 80%)')

## 6. Time-Series Cross-Validation

> *"Expanding window CV avoids look-ahead bias and provides robust performance estimates."*  
> — Petropoulos et al. (2022), Section 2.7.5

In [ ]:
# 6.1 Time-Series CV with expanding window
tscv = TimeSeriesSplit(n_splits=3)
X_all = fg[available_feats].values
y_all = fg[TARGET].values

cv_results = {'LightGBM': [], 'XGBoost': [], 'CatBoost': []}

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_all)):
    X_tr, X_va = X_all[train_idx], X_all[val_idx]
    y_tr, y_va = y_all[train_idx], y_all[val_idx]
    
    # LightGBM
    m = lgb.LGBMRegressor(**{**lgb_params, 'n_estimators': 200, 'verbose': -1})
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
    pred = np.clip(m.predict(X_va), 0, None)
    cv_results['LightGBM'].append(wape(y_va, pred))
    
    # XGBoost
    m = xgb.XGBRegressor(**{**xgb_params, 'n_estimators': 200})
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    pred = np.clip(m.predict(X_va), 0, None)
    cv_results['XGBoost'].append(wape(y_va, pred))
    
    # CatBoost
    m = CatBoostRegressor(**{**cat_params, 'iterations': 200})
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
    pred = np.clip(m.predict(X_va), 0, None)
    cv_results['CatBoost'].append(wape(y_va, pred))
    
    print(f'Fold {fold+1}: LGB={cv_results["LightGBM"][-1]:.4f}, XGB={cv_results["XGBoost"][-1]:.4f}, CAT={cv_results["CatBoost"][-1]:.4f}')

print('\n=== Time-Series CV Summary (WAPE) ===')
for name, scores in cv_results.items():
    print(f'  {name}: mean={np.mean(scores):.4f} +/- {np.std(scores):.4f}')

## 7. Feature Importance

In [ ]:
# 7.1 LightGBM Feature Importance (gain-based)
importance = pd.DataFrame({
    'feature': available_feats,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(importance)*0.3)))
ax.barh(importance['feature'][:20], importance['importance'][:20])
ax.set_xlabel('Feature Importance (Split Count)')
ax.set_title('LightGBM — Top 20 Features')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
for _, row in importance.head(5).iterrows():
    print(f'  {row["feature"]}: {row["importance"]}')

## 8. Model Comparison Summary

In [ ]:
# 8.1 Results comparison table
results_df = pd.DataFrame(results_log)
display_cols = ['model', 'type', 'RMSE', 'MAE', 'WAPE', 'R2', 'Bias']
display_cols = [c for c in display_cols if c in results_df.columns]

print('=== Model Comparison (Validation Set) ===')
print(results_df[display_cols].to_string(index=False, float_format='%.4f'))

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, metric in enumerate(['WAPE', 'RMSE', 'R2']):
    if metric in results_df.columns:
        colors = ['#e74c3c' if t == 'Baseline' else '#3498db' if t == 'ML' else '#27ae60'
                  for t in results_df['type']]
        axes[i].barh(results_df['model'], results_df[metric], color=colors)
        axes[i].set_title(metric)
        if metric == 'WAPE':
            axes[i].axvline(0.10, color='green', linestyle='--', alpha=0.5, label='Target (10%)')
            axes[i].legend()

plt.suptitle('Model Performance Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Best model
best = results_df.loc[results_df['WAPE'].idxmin()]
print(f'\nBest model: {best["model"]} (WAPE: {best["WAPE"]:.4f})')

# Save results
results_df.to_csv(ENG_DIR / 'model_comparison_results.csv', index=False)
print(f'\nNext: Notebook 04 — Detailed Model Evaluation & Diagnostics')